In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS main.gold;

In [0]:
from pyspark.sql import functions as F

print("Processando a tabela: fat_vendas_comercial")

try:
    # Leitura das tabelas da camada Silver
    df_itens = spark.table("main.silver.fat_itens_pedidos")
    df_pedidos = spark.table("main.silver.fat_pedidos")
    df_produtos = spark.table("main.silver.dim_produtos")
    df_dolar = spark.table("main.silver.dim_cotacao_dolar")

    # Cruzamento (Join) dos dados
    # Juntamos itens com os pedidos para ter a data
    df_joined = df_itens.join(df_pedidos, "id_pedido", "inner")
    
    # Juntamos com os produtos para ter a categoria
    df_joined = df_joined.join(df_produtos, "id_produto", "inner")

    # Preparamos a data (sem horas) para cruzar com o dólar
    df_joined = df_joined.withColumn("data_pura_compra", F.to_date(F.col("data_compra")))
    df_joined = df_joined.join(df_dolar, df_joined.data_pura_compra == df_dolar.data_cotacao, "left")

    # Extração de Ano, Mês e Cálculo do Dólar por Item
    df_preparado = df_joined \
        .withColumn("ano_venda", F.year(F.col("data_compra"))) \
        .withColumn("mes_venda", F.month(F.col("data_compra"))) \
        .withColumn("receita_item_usd", F.col("preco_BRL") / F.col("valor_dolar"))

    # Agrupamento
    df_agrupado = df_preparado.groupBy("ano_venda", "mes_venda", "categoria_produto").agg(
        F.countDistinct("id_pedido").alias("total_pedidos"),
        F.count("id_item").alias("qtd_itens_vendidos"),
        F.sum("preco_BRL").alias("receita_total_brl"),
        F.sum("receita_item_usd").alias("receita_total_usd")
    )

    # Cálculo do Ticket Médio e Arredondamentos Finais
    df_final = df_agrupado \
        .withColumn("ticket_medio_brl", F.col("receita_total_brl") / F.col("total_pedidos")) \
        .select(
            "ano_venda",
            "mes_venda",
            "categoria_produto",
            "total_pedidos",
            "qtd_itens_vendidos",
            F.round("receita_total_brl", 2).alias("receita_total_brl"),
            F.round("receita_total_usd", 2).alias("receita_total_usd"),
            F.round("ticket_medio_brl", 2).alias("ticket_medio_brl")
        ) \
        .orderBy("ano_venda", "mes_venda", "receita_total_brl", ascending=[True, True, False])

    # Guardar na camada Gold como tabela Delta
    destino = "main.gold.fat_vendas_comercial"
    df_final.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(destino)

    print(f"Sucesso: Tabela {destino} gerada! A área comercial já pode fazer os dashboards.")

except Exception as e:
    print(f"Erro ao processar fat_vendas_comercial: {e}")

In [0]:
display(spark.table("main.gold.fat_vendas_comercial"))

In [0]:
from pyspark.sql import functions as F

print("Gerando Rankings Comerciais...\n")

try:
    # Leitura das tabelas necessárias
    df_itens = spark.table("main.silver.fat_itens_pedidos")
    df_produtos = spark.table("main.silver.dim_produtos")

    # Cruzamento (Join) para trazer o Nome e a Categoria
    df_vendas_produtos = df_itens.join(df_produtos, "id_produto", "inner")

    # Agrupamento e cálculo da quantidade vendida
    df_ranking_base = df_vendas_produtos.groupBy("nome_produto", "categoria_produto").agg(
        F.count("id_item").alias("quantidade_vendida")
    )

    # Cálculo e Exibição: Top 5 MAIS Vendidos
    # Ordenamos de forma decrescente (desc) e limitamos a 5 linhas
    df_top5_mais = df_ranking_base.orderBy(F.col("quantidade_vendida").desc()).limit(5)
    
    print("Top 5 Produtos MAIS Vendidos:")
    display(df_top5_mais)

    # Cálculo e Exibição: Top 5 MENOS Vendidos
    # Ordenamos de forma ascendente (asc) e limitamos a 5 linhas
    df_top5_menos = df_ranking_base.orderBy(F.col("quantidade_vendida").asc()).limit(5)
    
    print("Top 5 Produtos MENOS Vendidos:")
    display(df_top5_menos)

except Exception as e:
    print(f"Erro ao gerar os rankings: {e}")

In [0]:
from pyspark.sql import functions as F

print("Processando a tabela: fat_avaliacoes_clientes...")

try:
    # Leitura das tabelas necessárias da camada Silver
    df_avaliacoes = spark.table("main.silver.fat_avaliacoes_pedidos")
    df_itens = spark.table("main.silver.fat_itens_pedidos")
    df_produtos = spark.table("main.silver.dim_produtos")
    df_vendedores = spark.table("main.silver.dim_vendedores")

    # Mega Join para juntar todas as pontas
    df_completo = df_avaliacoes \
        .join(df_itens, "id_pedido", "inner") \
        .join(df_produtos, "id_produto", "inner") \
        .join(df_vendedores, "id_vendedor", "inner")

    # Deduplicação inteligente
    # Garante que 1 pedido com vários itens iguais conte a avaliação apenas 1 vez para o vendedor
    df_base_avaliacoes = df_completo.select(
        "id_avaliacao", 
        "nota_avaliacao", 
        "categoria_produto", 
        "nome_vendedor", 
        "estado"
    ).distinct()

    # Agrupamento (Granularidade) e Cálculo das Métricas Base
    df_agrupado = df_base_avaliacoes.groupBy("categoria_produto", "nome_vendedor", "estado").agg(
        F.count("id_avaliacao").alias("total_avaliacoes"),
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
        # Somamos 1 se a nota for >= 4, senão 0
        F.sum(F.when(F.col("nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
        # Somamos 1 se a nota for <= 2, senão 0
        F.sum(F.when(F.col("nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas")
    )

    # Cálculo do Percentual de Satisfação
    df_final = df_agrupado.withColumn(
        "percentual_satisfacao",
        F.round((F.col("total_avaliacoes_positivas") / F.col("total_avaliacoes")) * 100, 2)
    )

    # Ordenação opcional para facilitar a leitura (dos mais bem avaliados para os piores)
    df_final = df_final.orderBy(F.col("percentual_satisfacao").desc(), F.col("total_avaliacoes").desc())

    # Guardar na camada Gold como tabela Delta
    destino = "main.gold.fat_avaliacoes_clientes"
    df_final.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(destino)

    print(f"Sucesso: Tabela {destino} gerada com as métricas de qualidade de parceiros!")

except Exception as e:
    print(f"Erro ao processar fat_avaliacoes_clientes: {e}")

In [0]:
display(spark.table("main.gold.fat_avaliacoes_clientes"))

In [0]:
from pyspark.sql import functions as F

print("Gerando Rankings de Qualidade...\n")

try:
    # Lendo as tabelas da Silver (Para garantir o cálculo da média)
    df_avaliacoes = spark.table("main.silver.fat_avaliacoes_pedidos")
    df_itens = spark.table("main.silver.fat_itens_pedidos")
    df_produtos = spark.table("main.silver.dim_produtos")
    df_vendedores = spark.table("main.silver.dim_vendedores")

    # Join base: Avaliações + Itens
    df_base = df_avaliacoes.join(df_itens, "id_pedido", "inner")

    # construção do Ranking de Produtos
    # deduplicacao para não contar a mesma avaliação várias vezes se o cliente comprou 2 itens iguais
    df_prod_base = df_base.join(df_produtos, "id_produto", "inner") \
        .select("id_avaliacao", "nota_avaliacao", "nome_produto").distinct()

    df_rank_produtos = df_prod_base.groupBy("nome_produto").agg(
        F.round(F.avg("nota_avaliacao"), 2).alias("nota_media"),
        F.count("id_avaliacao").alias("total_avaliacoes")
    )

    # construção do Ranking de Vendedores
    df_vend_base = df_base.join(df_vendedores, "id_vendedor", "inner") \
        .select("id_avaliacao", "nota_avaliacao", "nome_vendedor").distinct()

    df_rank_vendedores = df_vend_base.groupBy("nome_vendedor").agg(
        F.round(F.avg("nota_avaliacao"), 2).alias("nota_media"),
        F.count("id_avaliacao").alias("total_avaliacoes")
    )

    # Aplicando os critérios e exibindo via display()
    
    print("O Produto MAIS bem avaliado:")
    # Nota Alta (desc) -> Desempate por Volume (desc)
    df_prod_mais = df_rank_produtos.orderBy(F.col("nota_media").desc(), F.col("total_avaliacoes").desc()).limit(1)
    display(df_prod_mais)

    print("O Produto MENOS bem avaliado:")
    # Nota Baixa (asc) -> Desempate por Volume (desc) - Quanto mais notas baixas pior é.
    df_prod_menos = df_rank_produtos.orderBy(F.col("nota_media").asc(), F.col("total_avaliacoes").desc()).limit(1)
    display(df_prod_menos)

    print("O Vendedor MAIS bem avaliado:")
    df_vend_mais = df_rank_vendedores.orderBy(F.col("nota_media").desc(), F.col("total_avaliacoes").desc()).limit(1)
    display(df_vend_mais)

    print("O Vendedor MENOS bem avaliado:")
    df_vend_menos = df_rank_vendedores.orderBy(F.col("nota_media").asc(), F.col("total_avaliacoes").desc()).limit(1)
    display(df_vend_menos)

except Exception as e:
    print(f"Erro ao gerar os rankings de qualidade: {e}")